# TCC WikiArt — ResNet-50 no Google Colab

Roteiro oficial da **Etapa 7** do projeto.

Este notebook permite, a partir de uma sessão nova do Colab:

- verificar GPU;
- clonar o GitHub;
- instalar dependências sem substituir o PyTorch CUDA do Colab;
- preparar o cache do WikiArt;
- montar o Google Drive;
- persistir checkpoints;
- validar os manifestos;
- executar smoke tests;
- treinar a cabeça da ResNet-50;
- fazer fine-tuning;
- recuperar métricas dos checkpoints;
- gerar as curvas oficiais;
- fazer backup dos resultados pequenos.

## Configuração oficial

- ResNet-50 + `IMAGENET1K_V2`
- entrada 256×256
- 4 classes: Renaissance, Baroque, Impressionism, Cubism
- seed 42
- batch size 32
- 8 patches por pintura
- cabeça: 3 épocas, LR `1e-3`
- fine-tuning: 3 épocas
- LR backbone `1e-5`
- LR `fc` `1e-4`
- weight decay `1e-4`
- melhor checkpoint: menor validation loss

> Compartilhar este notebook não compartilha VM, cache do Hugging Face, GPU ou arquivos privados do Google Drive.

## 0. Antes de executar

No Colab, selecione GPU em **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**.

O experimento oficial foi executado em uma **Tesla T4**.

As células demoradas usam flags `RUN_* = False` para evitar execuções acidentais.

In [ ]:
import sys
import os
import subprocess
from pathlib import Path

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ATENÇÃO: ative uma GPU no ambiente de execução do Colab.")

## 1. Clonar o repositório

Em sessão nova, o projeto será clonado em `/content/tcc_wikiart`.

Se a pasta já existir, nenhum `pull` automático é feito para evitar perda de resultados locais.

In [ ]:
REPO_URL = "https://github.com/fbrum44/tcc_wikiart.git"
REPO_DIR = Path("/content/tcc_wikiart")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )
    print("Repositório clonado.")
else:
    print("Repositório já existe.")

os.chdir(REPO_DIR)

print("Diretório atual:", Path.cwd())

subprocess.run(
    ["git", "log", "-1", "--oneline"],
    check=False,
)

subprocess.run(
    ["git", "status", "--short"],
    check=False,
)

In [ ]:
RUN_GIT_PULL = False

if RUN_GIT_PULL:

    status = subprocess.run(
        ["git", "status", "--porcelain"],
        cwd=REPO_DIR,
        text=True,
        capture_output=True,
        check=True,
    ).stdout.strip()

    if status:
        raise RuntimeError(
            "Existem alterações locais no Colab.\n"
            "Preserve-as antes de executar git pull.\n\n"
            + status
        )

    subprocess.run(
        ["git", "pull", "--rebase", "origin", "main"],
        cwd=REPO_DIR,
        check=True,
    )

else:
    print("Pull não executado.")

## 2. Dependências

O Colab já fornece `torch` e `torchvision` com suporte CUDA.  
Por isso, nesta etapa não reinstalamos essas bibliotecas pelo `requirements.txt`.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "datasets",
        "opencv-python",
        "pytest",
    ],
    check=True,
)

import cv2
import datasets
import numpy as np
import pandas as pd
import torchvision
import matplotlib

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("datasets:", datasets.__version__)
print("opencv:", cv2.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Cache do Hugging Face e WikiArt

O dataset original é `huggan/wikiart`.

O split chamado `train` no Hugging Face é apenas o split original da fonte.  
Os nossos splits experimentais de treino/validação/teste estão nos manifestos do GitHub.

In [ ]:
os.environ["HF_HOME"] = "/content/hf_cache"

print("HF_HOME =", os.environ["HF_HOME"])

In [ ]:
from datasets import load_dataset

RUN_PREPARE_WIKIART = True

if RUN_PREPARE_WIKIART:

    wikiart = load_dataset(
        "huggan/wikiart",
        split="train",
        keep_in_memory=False,
    )

    print(
        "Imagens disponíveis no WikiArt:",
        len(wikiart),
    )

    EXPECTED_SIZE = 81444

if len(wikiart) != EXPECTED_SIZE:
    raise RuntimeError(
        "O WikiArt atual não corresponde ao dataset "
        "utilizado na curadoria.\n"
        f"Esperado: {EXPECTED_SIZE}\n"
        f"Encontrado: {len(wikiart)}\n\n"
        "Não continue o experimento antes de investigar."
    )

## 4. Google Drive e checkpoints

Checkpoints devem sobreviver à VM temporária.

Estrutura:

```text
TCC_WikiArt_Compartilhado/
├── models/
└── outputs/
    └── resnet50/
```

`models/` do repositório vira um link simbólico para o Drive.  
`outputs/` permanece local ao repositório porque CSV/JSON/PNG pequenos podem ser versionados.

Compartilhe TCC_WikiArt_Compartilhado com seus colegas como Editor.

Cada colega deve adicionar um atalho diretamente em Meu Drive com exatamente o nome:
TCC_WikiArt_Compartilhado

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
SHARED_FOLDER_NAME = "TCC_WikiArt_Compartilhado"

DRIVE_ROOT = (
    Path("/content/drive/MyDrive")
    / SHARED_FOLDER_NAME
)

DRIVE_MODELS = (
    DRIVE_ROOT
    / "models"
)

DRIVE_OUTPUTS = (
    DRIVE_ROOT
    / "outputs"
    / "resnet50"
)

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        f"Pasta compartilhada não encontrada:\n"
        f"{DRIVE_ROOT}\n\n"
        "Adicione um atalho de "
        "'TCC_WikiArt_Compartilhado' "
        "diretamente em Meu Drive."
    )

DRIVE_MODELS.mkdir(
    parents=True,
    exist_ok=True,
)

DRIVE_OUTPUTS.mkdir(
    parents=True,
    exist_ok=True,
)

# Testar permissão de escrita
teste = DRIVE_ROOT / ".teste_colab"

try:
    teste.write_text(
        "ok",
        encoding="utf-8",
    )
    teste.unlink()

except Exception as exc:
    raise PermissionError(
        "A pasta foi encontrada, mas "
        "não há permissão de escrita. "
        "Confirme acesso de Editor."
    ) from exc

print("Pasta compartilhada:", DRIVE_ROOT)
print("Models:", DRIVE_MODELS)
print("Outputs:", DRIVE_OUTPUTS)

In [ ]:
LOCAL_MODELS = (
    REPO_DIR
    / "models"
)

if LOCAL_MODELS.is_symlink():

    destino = (
        LOCAL_MODELS.resolve()
    )

    if destino != DRIVE_MODELS.resolve():

        print(
            "models/ apontava para:",
            destino,
        )

        LOCAL_MODELS.unlink()

        os.symlink(
            DRIVE_MODELS,
            LOCAL_MODELS,
            target_is_directory=True,
        )

        print(
            "Link corrigido."
        )

    else:
        print(
            "Link já está correto."
        )

elif LOCAL_MODELS.exists():

    if (
        LOCAL_MODELS.is_dir()
        and not any(
            LOCAL_MODELS.iterdir()
        )
    ):
        LOCAL_MODELS.rmdir()

        os.symlink(
            DRIVE_MODELS,
            LOCAL_MODELS,
            target_is_directory=True,
        )

    else:
        raise RuntimeError(
            "Existe um models/ local "
            "com conteúdo. Não será apagado."
        )

else:

    os.symlink(
        DRIVE_MODELS,
        LOCAL_MODELS,
        target_is_directory=True,
    )

# Ignorar localmente o symlink models/ no Git do Colab.
exclude_path = (
    REPO_DIR
    / ".git"
    / "info"
    / "exclude"
)

conteudo = (
    exclude_path.read_text(
        encoding="utf-8"
    )
    if exclude_path.exists()
    else ""
)

if "/models" not in conteudo.splitlines():
    with exclude_path.open(
        "a",
        encoding="utf-8",
    ) as arquivo:
        arquivo.write("\n/models\n")

print(
    "Symlink models/ ignorado "
    "localmente pelo Git."
)

print()
print("models local:")
print(LOCAL_MODELS)

print()
print("aponta para:")
print(LOCAL_MODELS.resolve())

In [ ]:
for nome in [
    "resnet50_head_best.pt",
    "resnet50_finetune_best.pt",
]:

    caminho = (
        LOCAL_MODELS
        / nome
    )

    print(
        nome,
        "->",
        "OK" if caminho.exists()
        else "NÃO ENCONTRADO",
    )

    if caminho.exists():
        print(
            "  tamanho:",
            round(
                caminho.stat().st_size
                / (1024 ** 2),
                2,
            ),
            "MB",
        )

## 5. Verificar os splits e manifestos

Esperado:

- treino: 7.000 pinturas;
- validação: 1.500;
- teste: 1.500.

O conjunto de teste permanece reservado durante a Etapa 7.

In [ ]:
META = (
    REPO_DIR
    / "data"
    / "metadata"
)

for split in [
    "train",
    "validation",
    "test",
]:

    csv_path = (
        META
        / f"{split}.csv"
    )

    patches_path = (
        META
        / f"patches_{split}.csv.gz"
    )

    if not csv_path.exists():
        raise FileNotFoundError(
            csv_path
        )

    if not patches_path.exists():
        raise FileNotFoundError(
            patches_path
        )

    df = pd.read_csv(
        csv_path
    )

    print()
    print(
        split.upper(),
        "- pinturas:",
        len(df),
    )

    print(
        df[
            "Macro_Estilo"
        ]
        .value_counts()
        .sort_index()
    )

## 6. Validar a ResNet-50

Confere arquitetura, quatro saídas, normalização, entrada 256×256 e política de congelamento.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "src.models.resnet",
    ],
    cwd=REPO_DIR,
    check=True,
)

## 7. Smoke test da cabeça — opcional

Executa poucos batches reais. Serve para validar pipeline e checkpoint, não para medir desempenho.

In [ ]:
RUN_HEAD_SMOKE = False

if RUN_HEAD_SMOKE:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "src.training.train_resnet",
        ],
        cwd=REPO_DIR,
        check=True,
    )

else:
    print(
        "Smoke test não executado."
    )

## 8. Treinamento oficial da cabeça

Saídas:

```text
outputs/resnet50/head_history.csv
outputs/resnet50/head_config.json
models/resnet50_head_best.pt
```

Se o checkpoint da cabeça já estiver no Drive, esta etapa pode ser pulada.

In [ ]:
RUN_HEAD_TRAINING = False
ALLOW_OVERWRITE_OFFICIAL_CHECKPOINT = False

HEAD_CKPT = (
    LOCAL_MODELS
    / "resnet50_head_best.pt"
)

if RUN_HEAD_TRAINING:

    if (
        HEAD_CKPT.exists()
        and not ALLOW_OVERWRITE_OFFICIAL_CHECKPOINT
    ):
        raise RuntimeError(
            "O checkpoint oficial da cabeça já existe.\n"
            "Para realmente substituí-lo, defina:\n"
            "ALLOW_OVERWRITE_OFFICIAL_CHECKPOINT = True"
        )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "src.training.train_resnet_head",
            "--epochs", "3",
            "--train-k", "8",
            "--val-k", "8",
            "--batch-size", "32",
            "--num-workers", "2",
        ],
        cwd=REPO_DIR,
        check=True,
    )

else:
    print(
        "Treinamento da cabeça "
        "não executado."
    )

## 9. Smoke test do fine-tuning — opcional

Parte obrigatoriamente de `resnet50_head_best.pt`.  
As métricas do smoke não devem ser comparadas com as métricas oficiais.

In [ ]:
RUN_FINETUNE_SMOKE = False

if RUN_FINETUNE_SMOKE:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "src.training.finetune_resnet",
            "--batch-size", "32",
            "--train-k", "8",
            "--val-k", "8",
            "--num-workers", "2",
            "--smoke",
        ],
        cwd=REPO_DIR,
        check=True,
    )

else:
    print(
        "Smoke test do fine-tuning "
        "não executado."
    )

## 10. Fine-tuning oficial

- 3 épocas;
- backbone LR `1e-5`;
- `fc` LR `1e-4`;
- BatchNorm running statistics congeladas;
- melhor checkpoint por menor validation loss.

Saídas:

```text
outputs/resnet50/finetune_history.csv
outputs/resnet50/finetune_config.json
models/resnet50_finetune_best.pt
```

In [ ]:
RUN_FINETUNE = False
ALLOW_OVERWRITE_OFFICIAL_FINETUNE = False

FINETUNE_CKPT = (
    LOCAL_MODELS
    / "resnet50_finetune_best.pt"
)

if RUN_FINETUNE:

    if not (
        LOCAL_MODELS
        / "resnet50_head_best.pt"
    ).exists():
        raise FileNotFoundError(
            "Checkpoint da cabeça não encontrado."
        )

    if (
        FINETUNE_CKPT.exists()
        and not ALLOW_OVERWRITE_OFFICIAL_FINETUNE
    ):
        raise RuntimeError(
            "O checkpoint oficial de fine-tuning já existe.\n"
            "Para realmente substituí-lo, defina:\n"
            "ALLOW_OVERWRITE_OFFICIAL_FINETUNE = True"
        )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "src.training.finetune_resnet",
            "--epochs", "3",
            "--batch-size", "32",
            "--train-k", "8",
            "--val-k", "8",
            "--num-workers", "2",
        ],
        cwd=REPO_DIR,
        check=True,
    )

else:
    print(
        "Fine-tuning oficial "
        "não executado."
    )

## 11. Recuperar métricas do melhor checkpoint

Útil inclusive se a interface do Colab perder a conexão depois de uma época.

In [ ]:
FINETUNE_CKPT = (
    LOCAL_MODELS
    / "resnet50_finetune_best.pt"
)

if FINETUNE_CKPT.exists():

    checkpoint = torch.load(
        FINETUNE_CKPT,
        map_location="cpu",
        weights_only=False,
    )

    print(
        "Fine-tuning epoch:",
        checkpoint.get(
            "finetune_epoch"
        ),
    )

    print(
        "Época geral:",
        checkpoint.get(
            "overall_epoch"
        ),
    )

    print(
        "Train loss:",
        checkpoint.get(
            "train_loss"
        ),
    )

    print(
        "Train accuracy:",
        checkpoint.get(
            "train_accuracy"
        ),
    )

    print(
        "Validation loss:",
        checkpoint.get(
            "val_loss"
        ),
    )

    print(
        "Validation accuracy:",
        checkpoint.get(
            "val_accuracy"
        ),
    )

else:
    print(
        "Checkpoint final "
        "não encontrado."
    )

## 12. Consolidar histórico e gerar curvas

Gera:

```text
outputs/resnet50/resnet_history.csv
outputs/resnet50/resnet_loss.png
outputs/resnet50/resnet_accuracy.png
```

Épocas 1–3: cabeça.  
Épocas 4–6: fine-tuning.

In [ ]:
import matplotlib.pyplot as plt

base = (
    REPO_DIR
    / "outputs"
    / "resnet50"
)

head_path = (
    base
    / "head_history.csv"
)

fine_path = (
    base
    / "finetune_history.csv"
)

if not head_path.exists():
    raise FileNotFoundError(
        head_path
    )

if not fine_path.exists():
    raise FileNotFoundError(
        fine_path
    )

head = pd.read_csv(
    head_path
).copy()

fine = pd.read_csv(
    fine_path
).copy()

head["overall_epoch"] = (
    head["epoch"]
)

head["phase"] = "head"

fine["phase"] = "finetune"

cols = [
    "overall_epoch",
    "train_loss",
    "train_accuracy",
    "val_loss",
    "val_accuracy",
    "phase",
]

hist = pd.concat(
    [
        head[cols],
        fine[cols],
    ],
    ignore_index=True,
)

hist = (
    hist
    .sort_values(
        "overall_epoch"
    )
    .reset_index(
        drop=True
    )
)

hist.to_csv(
    base
    / "resnet_history.csv",
    index=False,
)

display(hist)

In [ ]:
plt.figure(
    figsize=(10, 5)
)

plt.plot(
    hist["overall_epoch"],
    hist["train_loss"],
    marker="o",
    label="Train loss",
)

plt.plot(
    hist["overall_epoch"],
    hist["val_loss"],
    marker="o",
    label="Validation loss",
)

plt.axvline(
    3.5,
    linestyle="--",
    linewidth=1,
)

plt.title(
    "ResNet-50 — Evolução da Loss"
)

plt.xlabel("Época")
plt.ylabel("Loss")

plt.xticks(
    hist["overall_epoch"]
)

plt.legend()
plt.grid(
    True,
    alpha=0.3,
)

plt.tight_layout()

plt.savefig(
    base
    / "resnet_loss.png",
    dpi=300,
)

plt.show()

In [ ]:
plt.figure(
    figsize=(10, 5)
)

plt.plot(
    hist["overall_epoch"],
    hist["train_accuracy"],
    marker="o",
    label="Train accuracy",
)

plt.plot(
    hist["overall_epoch"],
    hist["val_accuracy"],
    marker="o",
    label="Validation accuracy",
)

plt.axvline(
    3.5,
    linestyle="--",
    linewidth=1,
)

plt.title(
    "ResNet-50 — Evolução da Accuracy"
)

plt.xlabel("Época")
plt.ylabel("Accuracy")

plt.xticks(
    hist["overall_epoch"]
)

plt.legend()
plt.grid(
    True,
    alpha=0.3,
)

plt.tight_layout()

plt.savefig(
    base
    / "resnet_accuracy.png",
    dpi=300,
)

plt.show()

## 13. Backup de CSV/JSON/PNG no Drive

Os checkpoints já estão no Drive por meio de `models/`.  
Esta etapa copia também os resultados pequenos.

In [ ]:
import shutil

LOCAL_OUTPUTS = (
    REPO_DIR
    / "outputs"
    / "resnet50"
)

ARQUIVOS_OFICIAIS = [
    "head_config.json",
    "head_history.csv",
    "finetune_config.json",
    "finetune_history.csv",
    "resnet_history.csv",
    "resnet_loss.png",
    "resnet_accuracy.png",
]

for nome in ARQUIVOS_OFICIAIS:

    origem = (
        LOCAL_OUTPUTS
        / nome
    )

    destino = (
        DRIVE_OUTPUTS
        / nome
    )

    if origem.exists():

        shutil.copy2(
            origem,
            destino,
        )

        print(
            "Backup:",
            nome,
        )

    else:
        print(
            "Não encontrado:",
            nome,
        )

print(
    "\nBackup oficial concluído."
)

In [ ]:
subprocess.run(
    [
        "git",
        "status",
        "--short",
    ],
    cwd=REPO_DIR,
    check=False,
)